# Weapon Balance Analysis — Diva Ex Machina

This notebook analyzes data collected from the in-game shooting range of **Diva Ex Machina**, a 3D action game.

An automated test bot (`WeaponBalanceBot`) equips randomly generated weapon variants, fires at targets under different conditions, and records how long each target takes to kill.

The goal is to understand **how each weapon attribute affects weapon efficiency** under different situations, so that weapon balance can be judged with data instead of only by feel.

## Step 1: Understanding the Problem

### Game Design Problem

Weapon balance is usually tested by hand. It is slow, and the result depends on the tester's skill and feeling.

A weapon can also be strong in one situation and weak in another. For example, a shotgun is strong at close range but weak at long range.

### Machine Learning Question

**Given a weapon's attributes and the combat situation, can we predict how efficiently the weapon kills a target?**

- **Weapon attributes**: damage, fire rate, spread, recoil, magazine size, reload time, bullet speed, critical chance, critical multiplier, recoil control
- **Combat situation**: target distance, target HP, target movement speed

### Type of Machine Learning Problem

This is a **regression** problem, because the value we want to predict (kill time / damage per second) is a continuous number, not a category.

## Step 2: Loading the Dataset

In this step, we load the data recorded by the test bot.

The bot writes one CSV file per run into the `ShootingRangeData` folder, with file names like `bot_20261006_202741.csv`.
There may be more than one file, so we load **all** of them and combine them into one DataFrame.

Old or invalid runs are moved into the `ShootingRangeData/Old` subfolder, so they are **not** loaded here.

**Important:** Run this notebook from the project root folder (the same folder that contains `ShootingRangeData`).

In [ ]:
# Import pandas library to work with datasets
import pandas as pd

# Import glob to find all files that match a pattern
from glob import glob

# Import display function to show tables clearly in Jupyter Notebook
from IPython.display import display

# Folder where the test bot saves its CSV files
data_folder = "ShootingRangeData"

# Find every bot CSV file directly inside the folder (the "Old" subfolder is not included)
file_paths = sorted(glob(f"{data_folder}/bot_*.csv"))

# Print how many files were found and their names
print("Number of files found:", len(file_paths))
for path in file_paths:
    print(" -", path)

In [ ]:
# Read every CSV file and store each one as a DataFrame in a list
data_frames = []
for path in file_paths:
    one_file = pd.read_csv(path)
    # Remember which file each row came from (useful when checking problems later)
    one_file["source_file"] = path
    data_frames.append(one_file)

# Combine all files into one DataFrame
df = pd.concat(data_frames, ignore_index=True)

# Print a message to confirm that the dataset has been loaded successfully
print("Dataset loaded successfully!")

# Print the number of rows and columns in the dataset
print("Dataset shape:", df.shape)

# Display the first five rows of the dataset
display(df.head())

### Short Interpretation

Each row is **one engagement**: the bot spawns one target, fires at it until the target dies or the time limit is reached, and records the result together with the weapon's final attributes.

## Step 3: Initial Data Inspection

After loading the dataset, we inspect it before cleaning and modeling.

In this step, we will check:
the number of rows and columns, the first and last records, column names, data types, missing values, duplicate rows, and summary statistics.

In [ ]:
# Print a title for this output
print("Dataset Shape")
# Print a separator line to make the output clear
print("-" * 50)
# Show the number of rows and columns in the dataset
print(df.shape)

In [ ]:
# Print a title for this output
print("Last Five Rows of the Dataset")
# Print a separator line to make the output clear
print("-" * 50)
# Display the last five rows of the dataset
display(df.tail())

In [ ]:
# Print a title for this output
print("Column Names")
# Print a separator line to make the output clear
print("-" * 50)
# Display all column names as a list
print(df.columns.tolist())

In [ ]:
# Print a title for this output
print("Dataset Information")
# Print a separator line to make the output clear
print("-" * 50)
# Show column names, non-null values, and data types
df.info()

In [ ]:
# Print a title for this output
print("Missing Values in Each Column")
# Print a separator line to make the output clear
print("-" * 50)
# Count missing values in each column
missing = df.isnull().sum()
# Show only the columns that have missing values
print(missing[missing > 0] if missing.sum() > 0 else "No missing values.")

In [ ]:
# Print a title for this output
print("Duplicate Rows")
# Print a separator line to make the output clear
print("-" * 50)
# Count fully duplicated rows
print("Number of duplicate rows:", df.duplicated().sum())

In [ ]:
# Print a title for this output
print("Summary Statistics of Numerical Columns")
# Print a separator line to make the output clear
print("-" * 50)
# Show count, mean, standard deviation, min, quartiles, and max (transposed so each column is a row)
display(df.describe().T)

In [ ]:
# Print a title for this output
print("Number of Unique Values in Each Column")
# Print a separator line to make the output clear
print("-" * 50)
# Count how many different values each column has
# (a column with only 1 unique value carries no information for the model)
display(df.nunique().to_frame("unique_values"))

In [ ]:
# Print a title for this output
print("Rows per Base Weapon")
# Print a separator line to make the output clear
print("-" * 50)
# Count how many engagements were recorded for each base weapon
display(df["base_weapon"].value_counts().to_frame("rows"))

# Print a title for this output
print("Weapon Variants per Base Weapon")
# Print a separator line to make the output clear
print("-" * 50)
# Count how many different weapon variants were tested for each base weapon
display(df.groupby("base_weapon")["variant_id"].nunique().to_frame("variants"))

## Step 4: Data Cleaning

The dataset has no missing values and no duplicate rows, but not every column should be used by the model.

In this step, we sort the columns into groups and remove the ones that carry no useful information:

1. **Constant columns**: columns with only one value tell the model nothing.
2. **Derived recoil columns**: values the game calculates from other columns.
3. **Validity checks**: make sure the recorded results make sense.

At the end of this step, we will have a clean DataFrame called `df_clean` and clear lists of which columns are features, which are results, and which are only for identification.

### Step 4.1: Remove Constant Columns

Some columns have the same value in every row, for example `hands` (the bot always uses two hands) or `explosion_damage` (none of the tested weapons deal explosion damage).

A column that never changes cannot explain any difference in weapon performance, so we remove it.

We find these columns automatically instead of typing them by hand. This way, if a future run changes one of these settings, the column will be kept.

Identification columns such as `session_id` are protected: they may be constant when only one bot run is loaded, but they are needed later to tell runs apart.

In [ ]:
# Make a copy so the original loaded data stays unchanged
df_clean = df.copy()

# Identification columns are always kept, even if they are constant right now
# (with only one bot run, session_id has one value; with several runs it will not)
protected_columns = ["session_id", "variant_id", "source_file"]

# Find every other column that has only one unique value
constant_columns = [col for col in df_clean.columns
                    if col not in protected_columns and df_clean[col].nunique() <= 1]

# Print a title for this output
print("Constant Columns (only one value)")
# Print a separator line to make the output clear
print("-" * 50)
# Show each constant column and its single value
for col in constant_columns:
    print(f"{col:30} = {df_clean[col].iloc[0]}")

# Remove the constant columns
df_clean = df_clean.drop(columns=constant_columns)

# Print the new shape of the dataset
print()
print("Shape after removing constant columns:", df_clean.shape)

### Step 4.2: Remove Derived Recoil Columns

The game uses three numbers that a designer actually sets:

| Column | Meaning | Set by |
|---|---|---|
| `recoil_per_shooting` | how much the aim is pushed each time the trigger is pulled | weapon |
| `time_between_shooting` | time between trigger pulls (fire rate) | weapon |
| `recoil_control` | how well the player resists recoil | armor (gloves / helmet) |

From these three, the game **calculates** four more values:

- `recoil` = recoil_per_shooting ÷ time_between_shooting × 1000
- `recoil_ratio` = recoil ÷ recoil_control
- `max_deviation` and `deviation_decay_per_second` are looked up from a curve using `recoil_ratio`

The calculated values are very highly correlated with each other. If we keep all of them, the model splits the importance between them, and it becomes impossible to tell which **design decision** actually matters.

So we keep only the three values a designer controls, and let the Random Forest learn how they interact.

In [ ]:
# Columns the game calculates from recoil_per_shooting, time_between_shooting and recoil_control
derived_recoil_columns = ["recoil", "recoil_ratio", "max_deviation", "deviation_decay_per_second"]

# Print a title for this output
print("Correlation Between Recoil-Related Columns")
# Print a separator line to make the output clear
print("-" * 50)
# Show how strongly the recoil columns move together (1.0 = move together perfectly)
recoil_columns = ["recoil_per_shooting", "time_between_shooting", "recoil_control"] + derived_recoil_columns
display(df_clean[recoil_columns].corr().round(2))

# Remove the derived columns (only the ones that still exist)
df_clean = df_clean.drop(columns=[c for c in derived_recoil_columns if c in df_clean.columns])

# Print the new shape of the dataset
print("Shape after removing derived recoil columns:", df_clean.shape)

### Step 4.3: Validity Checks

Before trusting the data, we check that every recorded result is possible.

- `time_to_kill` must be greater than 0.
- `damage_fraction` (how much of the target's HP was removed) must be between 0 and 1.
- If the target was killed, `damage_fraction` must be 1.
- If the target was **not** killed, the engagement must have reached the time limit, and `damage_fraction` must be below 1.

Rows where the target was not killed are **not errors**. They mean the weapon could not kill the target within the time limit (for example, a shotgun at 80 m). These rows are important and will be kept.

#### Invalid engagements: target died before the bot fired

Rarely, a target dies **before the bot pulls the trigger** (`trigger_pulls` = 0, negative `time_to_kill`).
This happens when a slow bullet from the **previous** engagement is still flying and hits the newly spawned target during the short waiting time between engagements.
These rows do not measure the weapon being tested, so we remove them first.

#### Engagements with zero damage

In a few engagements the weapon fired until the time limit but **never hit once** (`damage_fraction` = 0).
This is a real result (the weapon is useless in that situation), but its effective DPS is 0, and the logarithm of 0 is undefined, so the model cannot use it.
We count these rows and remove them, and keep in mind that the model therefore never sees the "completely useless" case.

In [ ]:
# An engagement is invalid if the bot never fired, or the recorded time is not positive
invalid_mask = (df_clean["trigger_pulls"] == 0) | (df_clean["time_to_kill"] <= 0)

# Print a title for this output
print("Invalid Engagements (target died before the bot fired)")
# Print a separator line to make the output clear
print("-" * 50)
print("Number of invalid rows:", invalid_mask.sum())
if invalid_mask.sum() > 0:
    display(df_clean.loc[invalid_mask, ["base_weapon", "variant_id", "distance", "target_hp",
                                        "time_to_kill", "trigger_pulls", "bullet_speed"]])

# Remove them
df_clean = df_clean[~invalid_mask].reset_index(drop=True)
print("Shape after removing invalid engagements:", df_clean.shape)

# Engagements where not a single bullet hit (effective DPS = 0, log is undefined)
zero_damage_mask = df_clean["damage_fraction"] <= 0
print()
print("Engagements with zero damage (never hit the target)")
print("-" * 50)
print("Number of rows:", zero_damage_mask.sum())
if zero_damage_mask.sum() > 0:
    display(df_clean.loc[zero_damage_mask, ["base_weapon", "variant_id", "distance", "target_hp",
                                            "target_speed", "trigger_pulls", "spread"]])
df_clean = df_clean[~zero_damage_mask].reset_index(drop=True)
print("Shape after removing zero-damage engagements:", df_clean.shape)

In [ ]:
# Print a title for this output
print("Validity Checks")
# Print a separator line to make the output clear
print("-" * 50)

# Each check counts how many rows break the rule (0 = no problem)
checks = {
    "time_to_kill <= 0":                     (df_clean["time_to_kill"] <= 0).sum(),
    "damage_fraction outside 0~1":           (~df_clean["damage_fraction"].between(0, 1)).sum(),
    "killed but damage_fraction < 1":        ((df_clean["killed"] == 1) & (df_clean["damage_fraction"] < 1)).sum(),
    "not killed but damage_fraction == 1":   ((df_clean["killed"] == 0) & (df_clean["damage_fraction"] >= 1)).sum(),
}

# Show the result of each check
for name, count in checks.items():
    print(f"{name:40} {count}")

# Stop the notebook here if any check fails, so we never train on broken data
assert sum(checks.values()) == 0, "Some rows have impossible values. Check the data before continuing."
print()
print("All validity checks passed.")

In [ ]:
# Print a title for this output
print("Engagements That Reached the Time Limit (target not killed)")
# Print a separator line to make the output clear
print("-" * 50)
# Count how many engagements did not kill the target
not_killed = df_clean[df_clean["killed"] == 0]
print(f"{len(not_killed)} of {len(df_clean)} rows ({len(not_killed) / len(df_clean):.0%})")
print("Time limit used:", sorted(not_killed["time_to_kill"].round(2).unique()))
print()
# Show where they happened: by base weapon and distance
display(pd.crosstab(not_killed["base_weapon"], not_killed["distance"]))

### Short Interpretation

Most engagements that reached the time limit are long-range shots with the shotgun and the SMG. This matches what we expect: these weapons have wide spread, so many bullets miss at long distance.

These rows must **not** be deleted. If we removed them, the model would only see the cases where a weapon succeeded, and it would believe that weak weapons are stronger than they really are. In the next step, we will turn every row (killed or not) into one fair measure of performance.

### Step 4.4: Column Groups

Finally, we sort the remaining columns into groups. Only the **feature** columns will be given to the model.

- **Identification**: which run and which weapon variant a row belongs to. `variant_id` will be used later to split training and testing data by weapon.
- **Condition features**: the combat situation.
- **Weapon features**: the attributes a designer sets.
- **Results**: what happened in the engagement. These must **never** be used as features, because they contain the answer.
  `trigger_pulls` is also a result: the longer the fight lasted, the more times the trigger was pulled.
- **Baseline formula**: simple DPS formulas, used later to compare against the model.

`base_weapon` is not used as a feature. If the model knew the weapon's name, it could simply memorize "this is a shotgun" instead of learning from the attributes.

`bullets_per_shot` is kept as a weapon feature, but it was not randomized: it is 1 for the rifle and SMG, and 8 for the shotgun. Its importance should be read as "being a pellet weapon", not as the effect of changing the pellet count.

In [ ]:
# Columns that identify a row (not features)
id_columns = ["session_id", "variant_id", "base_weapon", "is_baseline", "slot_name", "source_file"]

# The combat situation
condition_features = ["distance", "target_hp", "target_speed"]

# Weapon attributes a designer sets
weapon_features = [
    "physical_damage", "crit_chance", "crit_multiplier", "bullets_per_shot",
    "time_between_shooting", "spread", "recoil_per_shooting", "recoil_control",
    "magazine_size", "reload_time", "bullet_speed",
]

# What happened in the engagement (never used as features)
result_columns = ["killed", "time_to_kill", "damage_fraction", "trigger_pulls"]

# Simple formulas used later as a baseline
baseline_columns = ["burst_dps", "sustained_dps", "baseline_ttk"]

# Keep only the columns that still exist after cleaning (constant columns were removed)
id_columns = [c for c in id_columns if c in df_clean.columns]
condition_features = [c for c in condition_features if c in df_clean.columns]
weapon_features = [c for c in weapon_features if c in df_clean.columns]

# Check that every remaining column belongs to exactly one group
all_groups = id_columns + condition_features + weapon_features + result_columns + baseline_columns
not_grouped = [c for c in df_clean.columns if c not in all_groups]

# Print a title for this output
print("Column Groups")
# Print a separator line to make the output clear
print("-" * 50)
print("Identification    :", id_columns)
print("Condition features:", condition_features)
print("Weapon features   :", weapon_features)
print("Results           :", result_columns)
print("Baseline formula  :", baseline_columns)
print()
print("Columns not in any group:", not_grouped if not_grouped else "none")
print()
print("Final shape of df_clean:", df_clean.shape)

## Step 5: Defining Features and Target

### 1. What Should the Model Predict?

The most natural target would be `time_to_kill`. But it has a problem: **16% of the engagements did not kill the target** within the time limit.

For those rows, `time_to_kill` only says "more than 30 seconds", not the real value. If we used it directly:

- deleting those rows would hide the worst results and make weak weapons look stronger;
- keeping them as "30 seconds" would pretend that they almost succeeded.

### 2. Effective DPS

Instead, we use **effective DPS** (damage per second that actually landed):

> **effective DPS = target HP × damage fraction ÷ time**

- If the target was killed, damage fraction = 1, so effective DPS = HP ÷ kill time.
- If the target was not killed, we still know how much HP was removed in 30 seconds.

This way, **every row** gives a fair measure of how well the weapon performed, and no data is thrown away.

Kill time can always be recovered from it: **kill time ≈ HP ÷ effective DPS**.

In [ ]:
# Import numpy for mathematical functions such as log
import numpy as np

# Calculate effective DPS for every row (killed or not)
df_clean["effective_dps"] = df_clean["target_hp"] * df_clean["damage_fraction"] / df_clean["time_to_kill"]

# Print a title for this output
print("Effective DPS — Summary")
# Print a separator line to make the output clear
print("-" * 50)
# Show the basic statistics of the new column
display(df_clean["effective_dps"].describe().to_frame().T)

### 3. Why We Use the Logarithm

Effective DPS covers a very wide range: a shotgun at 10 m can kill a target almost instantly (thousands of DPS), while the same shotgun at 80 m may deal only a few damage per second.

If the model learned the raw numbers, a 100 DPS mistake on a 2000 DPS row would look much worse than a 5 DPS mistake on a 10 DPS row, even though the second one is a much bigger mistake **in relative terms** (50% vs 5%).

Taking the **logarithm** fixes this: the model is judged on relative error, so close-range and long-range situations count equally.

We use `log_effective_dps` as the target `y`.

In [ ]:
# Take the natural logarithm of effective DPS
df_clean["log_effective_dps"] = np.log(df_clean["effective_dps"])

# Import matplotlib to draw charts
import matplotlib.pyplot as plt

# Create two charts side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: original effective DPS (very skewed)
axes[0].hist(df_clean["effective_dps"], bins=40, color="steelblue")
axes[0].set_title("Effective DPS (original)")
axes[0].set_xlabel("Effective DPS")
axes[0].set_ylabel("Number of engagements")

# Right: log of effective DPS (much more balanced)
axes[1].hist(df_clean["log_effective_dps"], bins=40, color="darkorange")
axes[1].set_title("log(Effective DPS)")
axes[1].set_xlabel("log(Effective DPS)")

# Adjust spacing and show the charts
plt.tight_layout()
plt.show()

### Short Interpretation

The original effective DPS is heavily skewed: most engagements are in the low range, with a long tail of very fast kills.
After taking the logarithm, the values are spread much more evenly, which makes it easier for the model to learn.

The log chart also shows two groups: a large peak on the right (close-range engagements where most bullets hit) and a wider, lower spread on the left (long-range or moving-target engagements where many bullets miss). The model's main job is to learn **which weapon attributes move an engagement from the left group to the right group**.

The very high values come from close-range kills that happened in a single volley (for example, a shotgun at 10 m). In that case the "time" is mostly the bullet's travel time, so effective DPS becomes very large. It is still correct in ranking terms: a faster kill always gives a higher value.

In [ ]:
# Print a title for this output
print("Median Effective DPS of the Original Weapons (baseline variants)")
# Print a separator line to make the output clear
print("-" * 50)
# Use only the unmodified weapons, so the table shows the real in-game weapons
baseline = df_clean[df_clean["is_baseline"] == 1]
# Rows: weapon and target speed. Columns: distance. Values: median effective DPS
table = baseline.pivot_table(index=["base_weapon", "target_speed"], columns="distance",
                             values="effective_dps", aggfunc="median")
display(table.round(1))

### Short Interpretation

This table is a quick sanity check before modeling. It should match what we saw when playing:

- the shotgun is strongest at 10 m and drops sharply with distance,
- the assault rifle keeps its performance best at long range,
- moving targets reduce performance, especially at long range.

If this table did not match the game, there would be a problem in the data collection, and we would stop here.

### 4. Defining X and y

- **X (features)**: the combat situation and the weapon attributes, the information the model is allowed to see.
- **y (target)**: `log_effective_dps`, the value the model must predict.
- **groups**: which weapon variant each row belongs to. This is not given to the model. It is used in the next step to make sure the model is tested on **weapons it has never seen**.

`session_id` and `variant_id` are combined into one group ID, because variant numbers restart from 0 in every bot run.

In [ ]:
# Features: combat situation + weapon attributes (defined in Step 4.4)
feature_columns = condition_features + weapon_features

# X: the information the model is allowed to see
X = df_clean[feature_columns]

# y: the value the model must predict
y = df_clean["log_effective_dps"]

# groups: one ID per weapon variant (variant numbers restart in every bot run, so add the session)
groups = df_clean["session_id"].astype(str) + "_" + df_clean["variant_id"].astype(str)

# Print a title for this output
print("Features and Target")
# Print a separator line to make the output clear
print("-" * 50)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of different weapon variants (groups):", groups.nunique())
print()
print("Feature columns:")
for col in feature_columns:
    print(" -", col)

## Step 6: Train/Test Split

### 1. Why We Split the Data

We train the model on one part of the data (**training set**) and evaluate it on another part it has never seen (**testing set**).
This tells us how well the model works on new data, not just how well it memorized the training data.

### 2. Why a Normal Random Split Is Not Enough Here

In class we used `train_test_split`, which picks **rows** at random.

In this dataset, each weapon variant appears in about 12 rows (one per test condition). With a random row split, the same weapon would appear in **both** the training set and the testing set. The model could then score well simply by **recognizing the weapon** ("I have seen these exact attributes before, they were fast at 10 m"), without learning how attributes affect performance.

This is a form of **data leakage**: the test score would look better than the model really is.

### 3. Splitting by Weapon Instead

We use `GroupShuffleSplit`, which splits by **group** instead of by row. Every row of a weapon variant goes to the same side.

The testing set then contains only **weapons the model has never seen**, which matches the real question: *"A designer creates a new weapon. Can we predict how it performs before testing it?"*

In [ ]:
# Import GroupShuffleSplit to split the data by weapon variant instead of by row
from sklearn.model_selection import GroupShuffleSplit

# Keep 20% of the weapon variants for testing; random_state makes the split repeatable
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# Get the row positions of the training and testing sets
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

# Create the training and testing sets
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]

# Print a title for this output
print("Train/Test Split by Weapon Variant")
# Print a separator line to make the output clear
print("-" * 50)
print(f"Training set: {len(X_train)} rows, {groups_train.nunique()} weapon variants")
print(f"Testing set : {len(X_test)} rows, {groups_test.nunique()} weapon variants")

In [ ]:
# Print a title for this output
print("Leakage Check")
# Print a separator line to make the output clear
print("-" * 50)

# Find weapon variants that appear in both sets (this must be empty)
shared_variants = set(groups_train) & set(groups_test)
print("Weapon variants in both training and testing set:", len(shared_variants))

# Stop the notebook if any weapon appears on both sides
assert len(shared_variants) == 0, "Data leakage: the same weapon is in both sets."
print("No weapon appears in both sets.")
print()

# Show how many variants of each base weapon ended up in each set
print("Weapon variants per base weapon:")
split_summary = pd.DataFrame({
    "train": df_clean.iloc[train_idx].groupby("base_weapon")["variant_id"].nunique(),
    "test":  df_clean.iloc[test_idx].groupby("base_weapon")["variant_id"].nunique(),
})
display(split_summary)

### Short Interpretation

No weapon variant appears in both sets, so the test score will measure how well the model predicts **new weapons**.

Every base weapon (SMG, assault rifle, shotgun) should appear in both sets. If one base weapon were missing from the training set, the model would have never seen anything like it. With more data from the bot, this becomes less of a risk.

## Step 7: Baseline — The Simple DPS Formula

### 1. What Is the Baseline?

Before training a machine learning model, we need something to compare it against.

Game designers often judge a weapon by its **DPS (damage per second)** calculated from the stat sheet:

> **DPS = damage × pellets × (1 + crit chance × (crit multiplier − 1)) ÷ time between shots** (× 2 hands)

This is stored in the `burst_dps` column. The formula assumes **every bullet hits**, so it predicts the same effective DPS at every distance and target speed.

### 2. Why This Baseline Matters

If the formula were already accurate, we would not need machine learning.
By measuring **where** the formula fails, we show exactly what the model has to learn.

We evaluate it on the **testing set only**, so that it is compared fairly with the model later.

In [ ]:
# Import evaluation metrics for regression
from sklearn.metrics import mean_absolute_error, r2_score

# The formula's prediction: effective DPS = stat-sheet DPS (in log scale, like y)
baseline_pred_test = np.log(df_clean["burst_dps"].iloc[test_idx])

# Mean absolute error in log scale
baseline_mae = mean_absolute_error(y_test, baseline_pred_test)

# R² score: 1.0 = perfect, 0 = no better than always predicting the average
baseline_r2 = r2_score(y_test, baseline_pred_test)

# Print a title for this output
print("Baseline (DPS Formula) on the Testing Set")
# Print a separator line to make the output clear
print("-" * 50)
print(f"MAE (log scale): {baseline_mae:.3f}")
print(f"Typical error  : the formula is off by about x{np.exp(baseline_mae):.2f}")
print(f"R² score       : {baseline_r2:.3f}")

### How to Read These Numbers

- **MAE (log scale)** is hard to read directly, so we convert it: `exp(MAE)` is the **typical factor** of error.
  For example, ×1.5 means the prediction is usually 50% too high or too low; ×3 means three times too high or too low.
- **R²** compares the prediction with simply guessing the average every time. 1.0 is perfect, 0 means "no better than the average", and a **negative** value means "worse than just guessing the average".

In [ ]:
# Build a table with the real value, the formula's prediction, and the error for each test row
baseline_errors = df_clean.iloc[test_idx][["distance", "target_speed", "base_weapon"]].copy()
baseline_errors["abs_log_error"] = np.abs(y_test.values - baseline_pred_test.values)

# Print a title for this output
print("Baseline Typical Error by Distance and Target Speed (x factor)")
# Print a separator line to make the output clear
print("-" * 50)
# Average error in each situation, converted to a "x factor" so it is easy to read
error_table = baseline_errors.pivot_table(index="target_speed", columns="distance",
                                          values="abs_log_error", aggfunc="mean")
display(np.exp(error_table).round(2))

In [ ]:
# Draw the formula's typical error at each distance
error_by_distance = np.exp(baseline_errors.groupby("distance")["abs_log_error"].mean())

plt.figure(figsize=(7, 4))
plt.bar(error_by_distance.index.astype(str) + " m", error_by_distance.values, color="indianred")
# A dashed line at x1 means "perfect prediction"
plt.axhline(1, color="gray", linestyle="--", linewidth=1)
plt.title("DPS Formula: Typical Error by Distance")
plt.xlabel("Target distance")
plt.ylabel("Typical error (x factor)")
plt.tight_layout()
plt.show()

### Short Interpretation

The DPS formula is reasonably close at **short range**, where almost every bullet hits.
As the distance grows, its error grows quickly, because it does not know about **spread, recoil, bullet speed, or target movement**: the very attributes that decide how many bullets actually hit.

This is the gap the machine learning model must close. In the next step, we will train a model and compare its error with this baseline, both overall and at each distance.

Note: the formula also tends to **overestimate** fast-firing weapons, because in the game the fire rate is limited by the frame rate (for example, the SMG is set to 14.3 shots per second but actually fires about 10).

## Step 8: Training a Random Forest Model

### 1. Why Random Forest?

A Random Forest combines many decision trees, each trained on a slightly different sample of the data, and averages their predictions.

It fits this problem well because:

- **Interactions**: the effect of one attribute depends on the situation. For example, spread barely matters at 10 m but matters a lot at 80 m. Trees learn this naturally by splitting on distance first, then on spread.
- **No scaling needed**: features with very different ranges (damage ≈ 10, recoil control ≈ 1000) can be used directly.
- **Stable with small data**: averaging many trees reduces overfitting compared with a single decision tree.

Since we are predicting a number, we use `RandomForestRegressor` instead of `RandomForestClassifier`.

In [ ]:
# Import the Random Forest model for regression
from sklearn.ensemble import RandomForestRegressor

# Create the model
rf_model = RandomForestRegressor(
    n_estimators=300,      # number of trees in the forest
    min_samples_leaf=3,    # each leaf must contain at least 3 rows (reduces overfitting)
    random_state=42,       # makes the result repeatable
    n_jobs=-1,             # use all CPU cores to train faster
)

# Train the model on the training set only
rf_model.fit(X_train, y_train)

# Print a message to confirm that training is finished
print("Random Forest model trained successfully!")

## Step 9: Evaluating the Random Forest Model

We evaluate the model the same way as the baseline: MAE (converted to a typical error factor) and R², on the **testing set**.

We also check the **training set**. If the model is much better on training data than on testing data, it is **overfitting**: it memorized the training weapons instead of learning general rules.

In [ ]:
# Predict on the training set and the testing set
rf_pred_train = rf_model.predict(X_train)
rf_pred_test = rf_model.predict(X_test)

# Calculate the metrics
rf_mae_train = mean_absolute_error(y_train, rf_pred_train)
rf_mae_test = mean_absolute_error(y_test, rf_pred_test)
rf_r2_train = r2_score(y_train, rf_pred_train)
rf_r2_test = r2_score(y_test, rf_pred_test)

# Put the baseline and the model side by side
results = pd.DataFrame({
    "MAE (log)":     [baseline_mae, rf_mae_train, rf_mae_test],
    "Typical error": [f"x{np.exp(baseline_mae):.2f}", f"x{np.exp(rf_mae_train):.2f}", f"x{np.exp(rf_mae_test):.2f}"],
    "R²":            [baseline_r2, rf_r2_train, rf_r2_test],
}, index=["DPS formula (test)", "Random Forest (train)", "Random Forest (test)"])

# Print a title for this output
print("Model Comparison")
# Print a separator line to make the output clear
print("-" * 50)
display(results.round(3))

In [ ]:
# Add the model's error to the per-situation table from Step 7
comparison = baseline_errors.copy()
comparison["rf_abs_log_error"] = np.abs(y_test.values - rf_pred_test)

# Typical error at each distance for both methods
by_distance = pd.DataFrame({
    "DPS formula":   np.exp(comparison.groupby("distance")["abs_log_error"].mean()),
    "Random Forest": np.exp(comparison.groupby("distance")["rf_abs_log_error"].mean()),
})

# Print a title for this output
print("Typical Error by Distance (x factor, 1.0 = perfect)")
# Print a separator line to make the output clear
print("-" * 50)
display(by_distance.round(2))

# Draw both methods side by side for each distance
ax = by_distance.plot(kind="bar", figsize=(8, 4), color=["indianred", "seagreen"], rot=0)
ax.set_yscale("log")
ax.axhline(1, color="gray", linestyle="--", linewidth=1)
ax.set_title("Typical Error by Distance: DPS Formula vs Random Forest")
ax.set_xlabel("Target distance (m)")
ax.set_ylabel("Typical error (x factor, log scale)")
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: real value vs predicted value for the testing set
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

# The perfect-prediction line (prediction = real value)
lims = [min(y_test.min(), baseline_pred_test.min(), rf_pred_test.min()) - 0.5,
        max(y_test.max(), baseline_pred_test.max(), rf_pred_test.max()) + 0.5]

for ax, pred, title, color in [
    (axes[0], baseline_pred_test, "DPS formula", "indianred"),
    (axes[1], rf_pred_test, "Random Forest", "seagreen"),
]:
    ax.scatter(y_test, pred, alpha=0.6, color=color)
    ax.plot(lims, lims, color="gray", linestyle="--", linewidth=1)
    ax.set_title(f"{title}: predicted vs real (testing set)")
    ax.set_xlabel("Real log(effective DPS)")
    ax.set_ylabel("Predicted log(effective DPS)")
    ax.set_xlim(lims)
    ax.set_ylim(lims)

plt.tight_layout()
plt.show()

### Short Interpretation

- Points on the dashed line are perfect predictions.
- The DPS formula's points form **horizontal bands**: for a given weapon it predicts the same value at every distance, while the real value drops as distance grows.
- The Random Forest's points should lie much closer to the line, especially for the low values (long range, moving targets), which is exactly where the formula fails.
- If the training error is much smaller than the testing error, the model is overfitting. With more weapon variants from the bot, the gap should shrink.

**An honest observation:** at **10 m**, the simple formula can be as good as, or even better than, the Random Forest. At close range almost every bullet hits, so the formula's arithmetic (damage × pellets × fire rate) is nearly correct. The Random Forest has to *learn* that multiplication from a limited number of weapons, and trees are not good at exact multiplication.

This suggests an improvement: let the formula do the arithmetic it is good at, and let the model learn only what the formula is missing (how many bullets actually land). We will try this later.

## Step 10: Cross-Validation by Weapon

The testing set from Step 6 contains only a few weapon variants, so the result depends on **which** weapons happened to be chosen.

**Cross-validation** gives a more reliable estimate: the data is split into 5 parts (by weapon), the model is trained 5 times, and each time a different part is used for testing. Every weapon is used for testing exactly once.

We use `GroupKFold` so that, in every round, the tested weapons were never seen during training.

In [ ]:
# Import GroupKFold and cross_val_predict for cross-validation by group
from sklearn.model_selection import GroupKFold, cross_val_predict

# Split the weapons into 5 groups of folds
group_kfold = GroupKFold(n_splits=5)

# For every row, get the prediction from the round where its weapon was in the test fold
cv_pred = cross_val_predict(rf_model, X, y, groups=groups, cv=group_kfold, n_jobs=-1)

# The formula's prediction for every row (it needs no training)
formula_pred_all = np.log(df_clean["burst_dps"])

# Compare both methods on all rows
cv_results = pd.DataFrame({
    "MAE (log)":     [mean_absolute_error(y, formula_pred_all), mean_absolute_error(y, cv_pred)],
    "R²":            [r2_score(y, formula_pred_all), r2_score(y, cv_pred)],
}, index=["DPS formula", "Random Forest (5-fold, by weapon)"])
cv_results["Typical error"] = [f"x{v:.2f}" for v in np.exp(cv_results["MAE (log)"])]

# Print a title for this output
print("Cross-Validation Results (every weapon tested once)")
# Print a separator line to make the output clear
print("-" * 50)
display(cv_results.round(3))

## Step 11: What If We Had Used a Normal Random Split?

To show why splitting by weapon matters, we repeat the cross-validation with a normal `KFold`, which splits **rows** at random. The same weapon then appears in both training and testing data.

If the score with the normal split is clearly better, the difference is the **leakage**: how much the model was "cheating" by recognizing weapons it had already seen.

In [ ]:
# Import KFold for a normal random split by row
from sklearn.model_selection import KFold

# Normal 5-fold cross-validation (rows are shuffled, weapons can appear on both sides)
random_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_pred_random = cross_val_predict(rf_model, X, y, cv=random_kfold, n_jobs=-1)

# Compare the two ways of splitting
leakage = pd.DataFrame({
    "MAE (log)": [mean_absolute_error(y, cv_pred_random), mean_absolute_error(y, cv_pred)],
    "R²":        [r2_score(y, cv_pred_random), r2_score(y, cv_pred)],
}, index=["Random split by row (leaky)", "Split by weapon (honest)"])
leakage["Typical error"] = [f"x{v:.2f}" for v in np.exp(leakage["MAE (log)"])]

# Print a title for this output
print("Random Split vs Split by Weapon")
# Print a separator line to make the output clear
print("-" * 50)
display(leakage.round(3))

### Short Interpretation

The random split usually gives a **better-looking** score, but that score is misleading: part of it comes from recognizing weapons that were already in the training data.

The split-by-weapon score is the honest one, because it measures what we actually want: predicting the performance of a **new** weapon that has never been tested.

## Step 12: Model B — Formula + Random Forest

### 1. The Idea

In Step 9 we saw that the DPS formula is accurate at close range, while the Random Forest struggles to learn the exact arithmetic (damage × pellets × fire rate) from a limited number of weapons.

**Model B** splits the work:

> **predicted effective DPS = formula DPS × predicted hit efficiency**

- The **formula** does the arithmetic it is good at: the stat-sheet DPS (`burst_dps`).
- The **Random Forest** predicts only the part the formula does not know: **hit efficiency**, the fraction of the stat-sheet DPS that actually lands.

Hit efficiency is affected by everything the formula ignores: spread, recoil, bullet travel time, target movement, reloads, and overkill.

### 2. Same Information, Different Job

Model B uses **exactly the same 14 features** as Model A (the pure Random Forest). It does not lose any information.
The only difference is what the trees have to learn.

In log scale, multiplication becomes addition, so:

> log(effective DPS) = log(formula DPS) + log(hit efficiency)

The Random Forest learns `log(hit efficiency)` = `y − log(formula DPS)`.

In [ ]:
# The formula's prediction in log scale for every row
log_formula_dps = np.log(df_clean["burst_dps"])

# Target for Model B: log(hit efficiency) = log(effective DPS) - log(formula DPS)
y_efficiency = y - log_formula_dps

# Hit efficiency in normal scale (1.0 = performs exactly like the stat sheet)
hit_efficiency = np.exp(y_efficiency)

# Print a title for this output
print("Hit Efficiency (effective DPS ÷ formula DPS)")
# Print a separator line to make the output clear
print("-" * 50)
display(hit_efficiency.describe().to_frame("hit_efficiency").T.round(3))

# Median hit efficiency by distance and target speed
print("Median hit efficiency by distance and target speed:")
display(df_clean.assign(hit_efficiency=hit_efficiency)
        .pivot_table(index="target_speed", columns="distance", values="hit_efficiency", aggfunc="median")
        .round(3))

### Short Interpretation

A hit efficiency of 1.0 would mean the weapon performs exactly like its stat sheet.
At 10 m it should be close to 1, and it should drop sharply with distance and target speed.
This is the pattern the Random Forest in Model B has to learn.

Values above 1 can appear at very short range, when a target dies in a single volley: the "time" is then only the bullet's travel time, which makes the measured DPS look higher than the stat sheet.

In [ ]:
# Create Model B with the same settings as Model A, so the comparison is fair
rf_model_b = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1,
)

# Train on the same training rows as Model A, but with the hit-efficiency target
rf_model_b.fit(X_train, y_efficiency.iloc[train_idx])

# Final prediction = formula + predicted hit efficiency (both in log scale)
model_b_pred_train = log_formula_dps.iloc[train_idx].values + rf_model_b.predict(X_train)
model_b_pred_test = log_formula_dps.iloc[test_idx].values + rf_model_b.predict(X_test)

# Print a message to confirm that training is finished
print("Model B trained successfully!")

## Step 13: Comparing the Three Approaches

We compare all three approaches on the **same testing set** and with the **same cross-validation by weapon**:

- **DPS formula**: stat sheet only
- **Model A**: pure Random Forest
- **Model B**: formula + Random Forest

In [ ]:
# Metrics on the training and testing sets for Model B
model_b_mae_train = mean_absolute_error(y_train, model_b_pred_train)
model_b_mae_test = mean_absolute_error(y_test, model_b_pred_test)

# All three approaches on the same testing set
test_comparison = pd.DataFrame({
    "MAE train (log)": [np.nan, rf_mae_train, model_b_mae_train],
    "MAE test (log)":  [baseline_mae, rf_mae_test, model_b_mae_test],
    "R² test":         [baseline_r2, rf_r2_test, r2_score(y_test, model_b_pred_test)],
}, index=["DPS formula", "Model A (pure RF)", "Model B (formula + RF)"])
test_comparison["Typical error (test)"] = [f"x{v:.2f}" for v in np.exp(test_comparison["MAE test (log)"])]

# Print a title for this output
print("Testing Set Comparison")
# Print a separator line to make the output clear
print("-" * 50)
display(test_comparison.round(3))

In [ ]:
# Cross-validation by weapon for Model B (same folds as Model A in Step 10)
cv_pred_b = log_formula_dps.values + cross_val_predict(
    rf_model_b, X, y_efficiency, groups=groups, cv=group_kfold, n_jobs=-1)

# All three approaches with cross-validation (every weapon tested once)
cv_comparison = pd.DataFrame({
    "MAE (log)": [mean_absolute_error(y, formula_pred_all),
                  mean_absolute_error(y, cv_pred),
                  mean_absolute_error(y, cv_pred_b)],
    "R²":        [r2_score(y, formula_pred_all), r2_score(y, cv_pred), r2_score(y, cv_pred_b)],
}, index=["DPS formula", "Model A (pure RF)", "Model B (formula + RF)"])
cv_comparison["Typical error"] = [f"x{v:.2f}" for v in np.exp(cv_comparison["MAE (log)"])]

# Print a title for this output
print("Cross-Validation Comparison (every weapon tested once)")
# Print a separator line to make the output clear
print("-" * 50)
display(cv_comparison.round(3))

In [ ]:
# Typical error at each distance, using the cross-validation predictions (all weapons)
cv_errors = pd.DataFrame({
    "distance": df_clean["distance"].values,
    "DPS formula": np.abs(y.values - formula_pred_all.values),
    "Model A (pure RF)": np.abs(y.values - cv_pred),
    "Model B (formula + RF)": np.abs(y.values - cv_pred_b),
})
cv_by_distance = np.exp(cv_errors.groupby("distance").mean())

# Print a title for this output
print("Typical Error by Distance (cross-validation, x factor)")
# Print a separator line to make the output clear
print("-" * 50)
display(cv_by_distance.round(2))

# Draw the three approaches side by side for each distance
ax = cv_by_distance.plot(kind="bar", figsize=(9, 4), rot=0,
                         color=["indianred", "seagreen", "royalblue"])
ax.set_yscale("log")
ax.axhline(1, color="gray", linestyle="--", linewidth=1)
ax.set_title("Typical Error by Distance (cross-validation)")
ax.set_xlabel("Target distance (m)")
ax.set_ylabel("Typical error (x factor, log scale)")
plt.tight_layout()
plt.show()

### Short Interpretation

- **Model A vs DPS formula**: the pure Random Forest is far better at long range, but may lose to the formula at 10 m.
- **Model B**: if the idea works, Model B should keep the formula's accuracy at close range **and** the Random Forest's accuracy at long range, giving the lowest error overall.
- The **cross-validation** result is more reliable than the single testing set, because every weapon is tested once instead of only a few.

These results will change as the bot collects more data. The final choice between Model A and Model B is made by running this same comparison on the complete dataset, not by intuition.

## Step 14: Explaining the Model with SHAP

### 1. What Is SHAP?

A Random Forest makes good predictions, but it does not directly tell us **why**.
**SHAP** (SHapley Additive exPlanations) breaks every single prediction into contributions from each feature:

> prediction = average prediction + contribution of distance + contribution of spread + ...

- A **positive** SHAP value means the feature pushed this prediction **up** (higher hit efficiency).
- A **negative** SHAP value means it pushed the prediction **down**.
- The **size** of the value shows how much that feature mattered for that row.

### 2. What We Are Explaining

We explain **Model B**, so SHAP tells us which attributes affect **hit efficiency**: the part of performance that the stat-sheet DPS formula cannot see.

Model B works in log scale, so SHAP values are in log units. To read them:
**+0.69 ≈ ×2 hit efficiency**, **−0.69 ≈ ×0.5 (half)**.

We explain all rows (`X`) to see how the model behaves across every situation.

In [ ]:
# Import the SHAP library
# (if it is not installed, run:  pip install shap)
import shap

# TreeExplainer is the fast, exact SHAP method for tree-based models such as Random Forest
explainer = shap.TreeExplainer(rf_model_b)

# Calculate one SHAP value per feature for every row
shap_values = explainer.shap_values(X)

# Put them in a DataFrame with the same column names as X, for easy analysis
shap_df = pd.DataFrame(shap_values, columns=feature_columns, index=X.index)

# Print a message to confirm the calculation is finished
print("SHAP values calculated:", shap_df.shape)

### 3. Overall Feature Importance

The bar chart ranks features by their **average absolute SHAP value**: how much, on average, each feature changes the predicted hit efficiency.

In [ ]:
# Bar chart: average impact of each feature (ignoring direction)
shap.summary_plot(shap_values, X, plot_type="bar", show=False)
plt.title("Overall Feature Importance (Model B: hit efficiency)")
plt.tight_layout()
plt.show()

### 4. Direction of Each Feature's Effect

The beeswarm plot shows **every row** as a dot:

- position left/right = the SHAP value (pushes hit efficiency down or up)
- color = the feature's value (red = high, blue = low)

For example, if `spread` shows red dots on the left, it means **high spread lowers hit efficiency**.

In [ ]:
# Beeswarm plot: direction and size of each feature's effect for every row
shap.summary_plot(shap_values, X, show=False)
plt.title("How Each Feature Affects Hit Efficiency")
plt.tight_layout()
plt.show()

### 5. Importance at Each Distance

This is the key question of the project: **does the importance of an attribute change with distance?**

We calculate the average absolute SHAP value of each feature **separately for each distance**.
Condition features (distance, HP, speed) are removed from this table, because we want to compare the **weapon attributes** a designer can change.

In [ ]:
# Average absolute SHAP value of each weapon attribute, at each distance
importance_by_distance = (shap_df[weapon_features].abs()
                          .groupby(df_clean.loc[X.index, "distance"])
                          .mean()
                          .T)

# Sort rows by overall importance so the most important attributes are on top
importance_by_distance = importance_by_distance.loc[
    importance_by_distance.mean(axis=1).sort_values(ascending=False).index]

# Print a title for this output
print("Average |SHAP| of Each Weapon Attribute by Distance")
# Print a separator line to make the output clear
print("-" * 50)
display(importance_by_distance.round(3))

# Heatmap: darker = more important
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(importance_by_distance.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(importance_by_distance.shape[1]))
ax.set_xticklabels([f"{d} m" for d in importance_by_distance.columns])
ax.set_yticks(range(importance_by_distance.shape[0]))
ax.set_yticklabels(importance_by_distance.index)
# Write the value inside each cell
for i in range(importance_by_distance.shape[0]):
    for j in range(importance_by_distance.shape[1]):
        value = importance_by_distance.values[i, j]
        ax.text(j, i, f"{value:.2f}", ha="center", va="center",
                color="white" if value > importance_by_distance.values.max() * 0.6 else "black", fontsize=9)
ax.set_title("Weapon Attribute Importance by Distance (Model B)")
ax.set_xlabel("Target distance")
fig.colorbar(im, ax=ax, label="Average |SHAP|")
plt.tight_layout()
plt.show()

### Short Interpretation

Read the heatmap **row by row**: a row that gets darker from left to right is an attribute that matters more at long range.

Things to look for:

- **Spread** should matter little at 10 m and a lot at 50–80 m.
- **Damage** and **fire rate** mostly affect the formula part (already handled), so in Model B they mainly capture effects like overkill and reloads.
- If an attribute stays light in every column, it has little influence on how many bullets land, and changing it is not an effective way to balance accuracy-related performance.

### 6. Testing a Hypothesis: Bullet Speed

Before looking at the data, we predicted:

> **Bullet speed only matters when the target is moving and far away.** Against a stationary target, every bullet eventually arrives, so bullet speed should hardly matter.

We check this by looking at the importance of `bullet_speed` for every combination of distance and target speed.

In [ ]:
# Average absolute SHAP value of bullet_speed for each distance and target speed
bullet_speed_importance = (shap_df["bullet_speed"].abs()
                           .groupby([df_clean.loc[X.index, "target_speed"],
                                     df_clean.loc[X.index, "distance"]])
                           .mean()
                           .unstack())

# Print a title for this output
print("Importance of bullet_speed (average |SHAP|)")
print("Rows: target speed (m/s)   Columns: distance (m)")
# Print a separator line to make the output clear
print("-" * 50)
display(bullet_speed_importance.round(3))

### Short Interpretation

If the hypothesis is correct, the values should be **smallest in the top-left** (close, stationary target) and **largest in the bottom-right** (far, fast-moving target).

If the pattern is different, that is also useful: it means some other game mechanic is involved, and we should investigate it instead of forcing the conclusion.

### 7. How Spread Changes Hit Efficiency

A **dependence plot** shows, for one feature, how its value (x-axis) relates to its SHAP value (y-axis).
Each dot is colored by distance, so we can see whether the effect of spread is different at different ranges.

In [ ]:
# Dependence plot for spread, colored by distance
shap.dependence_plot("spread", shap_values, X, interaction_index="distance", show=False)
plt.title("Effect of Spread on Hit Efficiency (color = distance)")
plt.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.tight_layout()
plt.show()

### Short Interpretation

- Points **above 0** mean that spread value increases hit efficiency; points **below 0** mean it decreases it.
- If the long-range points (one color) fall much more steeply than the short-range points (the other color), spread is a **long-range attribute**: it decides long-range performance but barely matters up close.
- A sudden drop at some spread value would be a **threshold**: below it the weapon still hits reliably, above it most bullets miss. That value would be very useful for weapon designers.

## Step 15: Why Do Fast-Firing Weapons Underperform?

### 1. The Question

In Step 14, weapons with a short `time_between_shooting` (fast fire rate) had **lower** hit efficiency than the formula expects.
There are two possible game mechanics behind this:

| Mechanism | How it works | Expected pattern |
|---|---|---|
| **A. Frame-rate limit** | A weapon can only fire on a new frame, so very fast weapons fire fewer shots than their stat sheet says. | The penalty is about the **same at every distance** (fewer bullets are fired, no matter where the target is). |
| **B. Recoil ceiling** | The game calculates recoil per second = recoil per shot ÷ time between shots. Faster fire → higher recoil per second → higher maximum aim deviation. | The penalty **grows with distance** (a small aim error hardly matters at 10 m, but misses at 80 m), and is stronger when **recoil per shot** is high. |

The overall SHAP importance in Step 14 cannot separate these, because it averages each feature's effect over all situations.
Here we look at **how the effect of fire rate changes with distance and with recoil**.

### 2. The Fast-Fire Penalty at Each Distance

We split the weapons into **fast** (time between shots < 0.1 s) and **slow** (≥ 0.2 s) and compare the average SHAP value of `time_between_shooting` at each distance.

- If only mechanism A exists, the gap between fast and slow should stay about the same.
- If mechanism B also exists, the gap should **grow** with distance.

In [ ]:
# Group each row by fire rate: fast, medium, or slow
tbs = df_clean.loc[X.index, "time_between_shooting"]
fire_group = pd.cut(tbs, bins=[0, 0.1, 0.2, np.inf], labels=["fast (<0.1 s)", "medium", "slow (≥0.2 s)"])

# Average SHAP value of time_between_shooting, by fire-rate group and distance
# (negative = fire rate lowers hit efficiency in that situation)
tbs_effect = (shap_df["time_between_shooting"]
              .groupby([fire_group, df_clean.loc[X.index, "distance"]], observed=True)
              .mean()
              .unstack())

# Print a title for this output
print("Average SHAP of time_between_shooting (rows: fire rate, columns: distance)")
# Print a separator line to make the output clear
print("-" * 50)
display(tbs_effect.round(3))

# The gap between slow and fast weapons at each distance
gap = tbs_effect.loc["slow (≥0.2 s)"] - tbs_effect.loc["fast (<0.1 s)"]
print("Gap (slow − fast) at each distance:")
display(gap.round(3).to_frame("gap").T)

In [ ]:
# Dependence plot: fire rate on the x-axis, its effect on the y-axis, colored by distance
shap.dependence_plot("time_between_shooting", shap_values, X, interaction_index="distance", show=False)
plt.title("Effect of Fire Rate on Hit Efficiency (color = distance)")
plt.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.tight_layout()
plt.show()

### 3. SHAP Interaction Values

Normal SHAP values give each feature one number. **SHAP interaction values** go one step further: they split each prediction into

- the **main effect** of each feature on its own, and
- the **interaction effect** of every pair of features (the extra effect that appears only when both are combined).

If mechanism B is real, `time_between_shooting` should have a clear interaction with **`distance`** and with **`recoil_per_shooting`**.

Note: interaction values take longer to compute than normal SHAP values (around a minute).

In [ ]:
# Calculate SHAP interaction values (one 14 x 14 matrix per row)
shap_interactions = explainer.shap_interaction_values(X)

# Position of time_between_shooting in the feature list
tbs_pos = feature_columns.index("time_between_shooting")

# Average absolute interaction between time_between_shooting and every other feature
# (interaction values are split in half between the two features, so we multiply by 2)
interaction_strength = pd.Series(
    2 * np.abs(shap_interactions[:, tbs_pos, :]).mean(axis=0),
    index=feature_columns,
).drop("time_between_shooting").sort_values(ascending=False)

# Main effect of time_between_shooting on its own (the diagonal)
main_effect = np.abs(shap_interactions[:, tbs_pos, tbs_pos]).mean()

# Print a title for this output
print("Interactions of time_between_shooting with Other Features")
# Print a separator line to make the output clear
print("-" * 50)
print(f"Main effect of time_between_shooting on its own: {main_effect:.3f}")
print()
display(interaction_strength.round(3).to_frame("average |interaction|"))

In [ ]:
# How the fire rate x distance interaction changes across distances, for fast and slow weapons
dist_pos = feature_columns.index("distance")
rps_pos = feature_columns.index("recoil_per_shooting")

interaction_table = pd.DataFrame({
    "tbs x distance": 2 * shap_interactions[:, tbs_pos, dist_pos],
    "tbs x recoil_per_shooting": 2 * shap_interactions[:, tbs_pos, rps_pos],
    "distance": df_clean.loc[X.index, "distance"].values,
    "fire": fire_group.values,
})

# Print a title for this output
print("Average Interaction Values (rows: fire rate, columns: distance)")
# Print a separator line to make the output clear
print("-" * 50)
for name in ["tbs x distance", "tbs x recoil_per_shooting"]:
    print(name)
    display(interaction_table.pivot_table(index="fire", columns="distance", values=name,
                                          aggfunc="mean", observed=True).round(3))

### Short Interpretation

- **Gap table (Section 2)**: the fast-fire penalty is already present at 10 m, which supports mechanism A (frame-rate limit). On average, the gap does **not** clearly grow with distance.
- **Interaction ranking (Section 3)**: the strongest interaction of fire rate is with **spread**, not with distance. This is a hint that the distance effect depends on how accurate the weapon is.

Averages over **all** weapons can hide a pattern that goes in opposite directions for different groups. So in the next section we check the data directly, **without the model**, separately for accurate and inaccurate weapons.

### 4. Checking Without the Model: Accurate vs Inaccurate Weapons

We compare the real (measured) hit efficiency of slow and fast weapons, **separately for each spread group**.
The value shown is the gap: `slow − fast` in log hit efficiency. A bigger gap means fast-firing weapons lose more.

In [ ]:
# Measured log hit efficiency (not the model's prediction)
measured = pd.DataFrame({
    "log_hit_efficiency": y_efficiency.loc[X.index].values,
    "fire": fire_group.values,
    "distance": df_clean.loc[X.index, "distance"].values,
    "spread_group": pd.cut(df_clean.loc[X.index, "spread"], bins=[0, 0.7, 1.5, 3, np.inf],
                           labels=["< 0.7°", "0.7–1.5°", "1.5–3°", "> 3°"]).values,
})

# Gap between slow and fast weapons, for each spread group and distance
gap_rows = {}
for spread_group, part in measured.groupby("spread_group", observed=True):
    table = part.pivot_table(index="fire", columns="distance", values="log_hit_efficiency",
                             aggfunc="mean", observed=True)
    if "fast (<0.1 s)" in table.index and "slow (≥0.2 s)" in table.index:
        gap_rows[f"{spread_group}  (n={len(part)})"] = table.loc["slow (≥0.2 s)"] - table.loc["fast (<0.1 s)"]

# Print a title for this output
print("Measured Gap (slow − fast) by Spread Group and Distance")
# Print a separator line to make the output clear
print("-" * 50)
display(pd.DataFrame(gap_rows).T.round(2))

### Short Interpretation

The two mechanisms appear in **different kinds of weapons**:

- **Accurate weapons (low spread)**: the fast-fire penalty **grows with distance**. When spread is small, recoil deviation is the main source of aim error, so the recoil ceiling (mechanism B) becomes important at long range.
- **Inaccurate weapons (high spread)**: the penalty **shrinks with distance**. Spread is already so large that most bullets miss at long range anyway; extra recoil deviation makes little difference.

Because the two groups move in **opposite directions**, the average over all weapons looks flat. This is why the overall SHAP values in Section 2 could not show the effect, and why the strongest interaction of fire rate is with **spread**.

**Conclusion so far:**
1. Mechanism A (frame-rate limit) affects all fast weapons at every distance.
2. Mechanism B (recoil ceiling) is real, but only matters for **accurate** weapons at **long range**: a three-way interaction (fire rate × distance × spread).

Some groups have few rows, so these numbers will be checked again with the complete dataset.
These are still **correlations**. To prove cause and effect, the final step is a controlled test with the bot: the same weapon with recoil per shot set to 0. Without recoil, mechanism B is switched off, so any remaining penalty must come from the frame-rate limit.

## Step 16: Learning Curve — Do We Have Enough Data?

### 1. The Question

The bot can keep collecting data for hours. But more data is only useful if it still makes the model better.

A **learning curve** answers this: we train the model with **25%, 50%, 75% and 100%** of the available weapons and measure the error each time.

- If the error is still **dropping** from 75% to 100%, more data would help → keep the bot running.
- If the curve has become **flat**, more data hardly changes the result → we can stop collecting.

### 2. How It Is Measured

For a fair result, we again use cross-validation **by weapon**:

1. Split the weapons into 5 folds with `GroupKFold`.
2. In each round, keep the test fold unchanged, but train only on a **random subset** of the training weapons (25%, 50%, ...).
3. Repeat with different random subsets and average the results, because small subsets can be lucky or unlucky.

We do this for both Model A (pure Random Forest) and Model B (formula + Random Forest).

In [ ]:
# Fractions of the training weapons to use, and how many times to repeat each one
fractions = [0.25, 0.5, 0.75, 1.0]
repeats = 3

# Random number generator with a fixed seed so the result is repeatable
rng = np.random.default_rng(42)

# Store the results of every round here
curve_rows = []

for fraction in fractions:
    for repeat in range(repeats):
        for fold, (train_rows, test_rows) in enumerate(group_kfold.split(X, y, groups=groups)):
            # Pick a random subset of the training weapons
            train_weapons = groups.iloc[train_rows].unique()
            n_keep = max(3, int(round(len(train_weapons) * fraction)))
            kept = rng.choice(train_weapons, size=n_keep, replace=False)
            subset = train_rows[groups.iloc[train_rows].isin(kept).values]

            # Model A: pure Random Forest
            model_a = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=repeat, n_jobs=-1)
            model_a.fit(X.iloc[subset], y.iloc[subset])
            pred_a = model_a.predict(X.iloc[test_rows])

            # Model B: formula + Random Forest
            model_b = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=repeat, n_jobs=-1)
            model_b.fit(X.iloc[subset], y_efficiency.iloc[subset])
            pred_b = log_formula_dps.iloc[test_rows].values + model_b.predict(X.iloc[test_rows])

            curve_rows.append({
                "fraction": fraction,
                "weapons_used": n_keep,
                "Model A": mean_absolute_error(y.iloc[test_rows], pred_a),
                "Model B": mean_absolute_error(y.iloc[test_rows], pred_b),
            })

curve = pd.DataFrame(curve_rows)

# Average over all folds and repeats
curve_summary = curve.groupby("fraction").agg(
    weapons_used=("weapons_used", "mean"),
    model_a_mae=("Model A", "mean"), model_a_std=("Model A", "std"),
    model_b_mae=("Model B", "mean"), model_b_std=("Model B", "std"),
)

# Print a title for this output
print("Learning Curve (cross-validation by weapon)")
# Print a separator line to make the output clear
print("-" * 50)
display(pd.DataFrame({
    "weapons used for training": curve_summary["weapons_used"].round(0).astype(int).values,
    "Model A typical error": [f"x{v:.2f}" for v in np.exp(curve_summary["model_a_mae"])],
    "Model B typical error": [f"x{v:.2f}" for v in np.exp(curve_summary["model_b_mae"])],
}, index=[f"{int(f * 100)}%" for f in curve_summary.index]))

In [ ]:
# Draw the learning curve: number of training weapons vs typical error
plt.figure(figsize=(8, 5))
for name, mae_col, std_col, color in [
    ("Model A (pure RF)", "model_a_mae", "model_a_std", "seagreen"),
    ("Model B (formula + RF)", "model_b_mae", "model_b_std", "royalblue"),
]:
    x_values = curve_summary["weapons_used"]
    mean = np.exp(curve_summary[mae_col])
    # Shaded band = one standard deviation across folds and repeats
    lower = np.exp(curve_summary[mae_col] - curve_summary[std_col])
    upper = np.exp(curve_summary[mae_col] + curve_summary[std_col])
    plt.plot(x_values, mean, marker="o", color=color, label=name)
    plt.fill_between(x_values, lower, upper, color=color, alpha=0.15)

plt.title("Learning Curve: More Weapons vs Prediction Error")
plt.xlabel("Number of weapon variants used for training")
plt.ylabel("Typical error (x factor, lower is better)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# How much did the last step (75% -> 100%) still improve each model?
improvement = pd.DataFrame({
    "Model A": [np.exp(curve_summary.loc[0.75, "model_a_mae"]) - np.exp(curve_summary.loc[1.0, "model_a_mae"])],
    "Model B": [np.exp(curve_summary.loc[0.75, "model_b_mae"]) - np.exp(curve_summary.loc[1.0, "model_b_mae"])],
}, index=["error reduction from 75% to 100% of the data"])

# Print a title for this output
print("Is the Curve Still Dropping?")
# Print a separator line to make the output clear
print("-" * 50)
display(improvement.round(3))
print("Rule of thumb: a reduction smaller than about 0.03 (3 percentage points of error) means the curve is nearly flat.")

### Short Interpretation

- The shaded bands show how much the result varies between folds. If the bands of two points overlap a lot, the difference between them is not reliable.
- **Still dropping**: more weapons would make the model more accurate → keep the bot running.
- **Flat**: the model has learned most of what this data can teach → stop collecting and spend the time on analysis.

A flat curve does not mean the model is perfect. It means the remaining error comes from things **more data of the same kind cannot fix**: random variation between engagements, game mechanics the features do not describe, or the limits of the model itself.

## Step 17: Which Attribute Matters Most in Each Situation?

### 1. The Question

Step 14 showed the importance of each attribute **for hit efficiency** (Model B). A designer often asks a simpler question:

> "In this situation, which stat changes the **real DPS** the most?"

For example: at 10 m against a moving target, or at 50 m against a target standing still.

### 2. How We Answer It

- We train a Random Forest on `log_effective_dps` directly (like Model A), so **damage and fire rate are included** in the explanation.
- We calculate SHAP values for every engagement.
- We group the engagements into the **12 situations** (4 distances × 3 target speeds) and average the absolute SHAP value of each attribute.
- We show each attribute's **share** of the total importance in that situation (in %), so the situations can be compared fairly.

Note: `bullets_per_shot` has only two values in this data (shotgun or not), so in practice it means **"is it a shotgun"**.

In [ ]:
# Train a Random Forest on real DPS (log scale) using all data, for explanation only
rf_scenario = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1)
rf_scenario.fit(X, y)

# SHAP values for every engagement
scenario_shap = shap.TreeExplainer(rf_scenario).shap_values(X)
scenario_abs = pd.DataFrame(np.abs(scenario_shap), columns=X.columns)[weapon_features]
scenario_abs["distance"] = X["distance"].values
scenario_abs["target_speed"] = X["target_speed"].values

# Average importance per situation, then turn it into a share (%) of the total
scenario_mean = scenario_abs.groupby(["distance", "target_speed"])[weapon_features].mean()
scenario_share = scenario_mean.div(scenario_mean.sum(axis=1), axis=0) * 100

# Print a title for this output
print("Most Important Attributes in Each Situation (share of importance, %)")
# Print a separator line to make the output clear
print("-" * 50)
top_rows = []
for (dist, speed), row in scenario_share.iterrows():
    top3 = row.sort_values(ascending=False).head(3)
    top_rows.append({
        "distance (m)": dist,
        "target speed (m/s)": speed,
        "1st": f"{top3.index[0]} ({top3.iloc[0]:.0f}%)",
        "2nd": f"{top3.index[1]} ({top3.iloc[1]:.0f}%)",
        "3rd": f"{top3.index[2]} ({top3.iloc[2]:.0f}%)",
    })
display(pd.DataFrame(top_rows))

In [ ]:
# Heatmap: share of importance of each attribute in each situation
plot_data = scenario_share.T
plot_data.columns = [f"{d} m / {s} m/s" for d, s in plot_data.columns]
order = plot_data.mean(axis=1).sort_values(ascending=False).index
plot_data = plot_data.loc[order]

fig, ax = plt.subplots(figsize=(13, 6))
image = ax.imshow(plot_data.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(plot_data.shape[1]))
ax.set_xticklabels(plot_data.columns, rotation=45, ha="right")
ax.set_yticks(range(plot_data.shape[0]))
ax.set_yticklabels(plot_data.index)
# Write the number in each cell (white text on dark cells)
for i in range(plot_data.shape[0]):
    for j in range(plot_data.shape[1]):
        value = plot_data.values[i, j]
        ax.text(j, i, f"{value:.0f}", ha="center", va="center",
                color="white" if value > 30 else "black", fontsize=9)
fig.colorbar(image, ax=ax, label="share of importance (%)")
ax.set_title("Which Attribute Drives Real DPS in Each Situation")
ax.set_xlabel("distance / target speed")
plt.tight_layout()
plt.show()

### Short Interpretation

- **Distance changes the answer; target speed hardly does.** The ranking at 0, 10 and 15 m/s is almost the same at each distance.
- **10 m:** being a shotgun (pellets per shot) and spread share the top place, and raw damage still matters (~15%).
- **20 m:** spread takes over clearly.
- **50 m and 80 m:** spread alone is about half of the importance, and recoil is second (~20%). Damage drops to ~7%.

For a designer this means: **at close range, balance with damage and pellets; at long range, balance with spread and recoil.** Changing damage on a long-range weapon has little effect on how it performs at range.